In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu, pearsonr

NL = chr(10)

# 1. Загрузка исторических курсов Clio Infra (1763-1913, T=151)
xl_fx = pd.ExcelFile('ExchangeRatestoUKPound_Compact.xlsx')
df_long = xl_fx.parse('Data Long Format')
df_long['year'] = pd.to_numeric(df_long['year'], errors='coerce')
df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

COUNTRIES_9 = ['Russia', 'Netherlands', 'Germany', 'France', 'Sweden', 'Austria', 'Spain', 'Portugal', 'Italy']
COUNTRIES_8_CLEAN = ['Russia', 'Netherlands', 'Germany', 'France', 'Austria', 'Spain', 'Portugal', 'Italy']

sub = df_long[(df_long['country.name'].isin(COUNTRIES_9)) & (df_long['year'].between(1763, 1913))].dropna(subset=['value'])
piv = sub.pivot_table(index='year', columns='country.name', values='value').reindex(range(1763, 1914))
piv_interp = piv.interpolate(method='linear', limit_direction='both')

# 2. Калибровка серебряного и золотого паритетов (Lindert 2006; Mironov 2010)
xl_gbp = pd.ExcelFile('Silver_value_of_1273-1977.xls')
d_gbp_raw = xl_gbp.parse('Ag content')
gbp_rows = []
for idx, row in d_gbp_raw.iterrows():
    try:
        yr = int(float(row.iat[0]))
        if 1763 <= yr <= 1913:
            mint_g = pd.to_numeric(row.iat[10], errors='coerce')
            mkt_g = pd.to_numeric(row.iat[11], errors='coerce')
            gbp_rows.append((yr, mint_g, mkt_g))
    except Exception:
        continue
df_gbp_ag = pd.DataFrame(gbp_rows, columns=['year', 'gbp_mint_g', 'gbp_mkt_g']).set_index('year')

df_par = pd.concat([piv['Russia'].rename('rub_mkt'), df_gbp_ag], axis=1)
df_par['rub_ag_g'] = np.where(df_par.index == 1763, 20.74, 18.00)
df_par['par_mint_ag'] = df_par['gbp_mint_g'] / df_par['rub_ag_g']
df_par['par_mkt_ag'] = df_par['gbp_mkt_g'] / df_par['rub_ag_g']
df_par['par_gold_1897'] = np.where(df_par.index >= 1897, 7.32238 / 0.77423, np.nan)
df_par['anchor_par'] = np.where(df_par.index >= 1897, df_par['par_gold_1897'], df_par['par_mint_ag'])
df_par['ratio_anchor'] = df_par['rub_mkt'] / df_par['anchor_par']

# 3. Спектральная декомпозиция, граф переходов и бутстрэп (Таблица 1)
def safe_zscore(M_arr):
    mean_v = np.mean(M_arr, axis=0)
    std_v = np.std(M_arr, axis=0, ddof=1)
    safe_std = np.where(std_v > 1e-12, std_v, 1.0)
    Z = (M_arr - mean_v) / safe_std
    Z[:, std_v <= 1e-12] = 0.0
    return Z

def run_spectral_audit(dlnX, yrs, country_names, label, n_boot=300, seed=42):
    Z = safe_zscore(dlnX)
    M, N = Z.shape
    U, s, Vt = np.linalg.svd(Z, full_matrices=False)
    u1, v1, v2 = np.take(U, 0, axis=1), np.take(Vt, 0, axis=0), np.take(Vt, 1, axis=0)
    var1 = (s.item(0)**2) / np.sum(s**2) * 100.0
    var2 = (s.item(1)**2) / np.sum(s**2) * 100.0
    pr_u1 = (1.0 / np.sum(u1**4) - 1.0) / (M - 1.0)
    pr_v1 = (1.0 / np.sum(v1**4) - 1.0) / (N - 1.0)

    rng = np.random.default_rng(seed)
    sub_M = int(np.floor(M * 0.8))
    b_pru, b_prv = [], []
    for _ in range(n_boot):
        idx = np.sort(rng.choice(M, size=sub_M, replace=False))
        Zb = safe_zscore(dlnX[idx, :])
        Ub, _, Vtb = np.linalg.svd(Zb, full_matrices=False)
        u1b, v1b = np.take(Ub, 0, axis=1), np.take(Vtb, 0, axis=0)
        b_pru.append((1.0 / np.sum(u1b**4) - 1.0) / (sub_M - 1.0))
        b_prv.append((1.0 / np.sum(v1b**4) - 1.0) / (N - 1.0))

    rus_idx = country_names.index('Russia')
    rus_share = (v1.item(rus_idx)**2) * 100.0
    print(f"{label:<45} | M={M:3d} | Var1={var1:4.1f}% | PR(u1)={pr_u1:.3f} [{np.percentile(b_pru, 2.5):.3f}, {np.percentile(b_pru, 97.5):.3f}] | PR(v1)={pr_v1:.3f} [{np.percentile(b_prv, 2.5):.3f}, {np.percentile(b_prv, 97.5):.3f}] | Russia={rus_share:5.2f}%")
    return u1, v1, v2, var1, var2, pr_u1, pr_v1

dln_full = np.diff(np.log(piv_interp[COUNTRIES_9].to_numpy(dtype=float)), axis=0)
yrs_full = np.arange(1764, 1914)
dln_1815 = np.diff(np.log(piv.loc[1815:1913, COUNTRIES_8_CLEAN].to_numpy(dtype=float)), axis=0)
yrs_1815 = np.arange(1816, 1914)

print("=== TABLE 1: SPECTRAL SUMMARY ===")
u1_f, v1_f, v2_f, var1_f, var2_f, _, _ = run_spectral_audit(dln_full, yrs_full, COUNTRIES_9, "Full European Panel (1764-1913, N=9)")
u1_15, v1_15, _, var1_15, _, pr_u1_15, _ = run_spectral_audit(dln_1815, yrs_1815, COUNTRIES_8_CLEAN, "Uninterpolated Clean (1816-1913, N=8)")
for p_name, y0, y1 in [
    ("Assignat and Napoleonic Era (1764-1839)", 1764, 1839),
    ("Silver and Credit Ruble Era (1840-1896)", 1840, 1896),
    ("Classical Gold Standard (1897-1913)", 1897, 1913)
]:
    m = (yrs_full >= y0) & (yrs_full <= y1)
    run_spectral_audit(dln_full[m, :], yrs_full[m], COUNTRIES_9, p_name)

# 4. Отрисовка публикационных графиков (300 dpi)
v1_plot = -v1_f
yrs_p = df_par.index.to_numpy()

# --- FIGURE 1 ---
fig1 = plt.figure(figsize=(15.5, 10.5), dpi=300)
plt.subplots_adjust(hspace=0.35, wspace=0.26)

ax1 = fig1.add_subplot(2, 2, 1)
ax1.plot(yrs_p, df_par['rub_mkt'], color='#c00000', lw=1.8, label='Market Exchange Rate (Rubles per £)')
ax1.plot(yrs_p, df_par['par_mint_ag'], color='#1f4e78', lw=1.3, linestyle='--', label='Mint Silver Parity (18.0g Ag/Rub)')
ax1.plot(yrs_p, df_par['par_mkt_ag'], color='#385723', lw=1.2, linestyle=':', label='London Market Silver Equivalent')
ax1.plot(yrs_p, df_par['par_gold_1897'], color='#ed7d31', lw=2.0, label='1897 Gold Standard Parity (9.46 Rub/£)')
ax1.annotate('1769 Assignats' + NL + 'Introduced', xy=(1769, 4.84), xytext=(1766, 11.8), arrowprops=dict(arrowstyle='->', color='black', lw=0.8), fontsize=8, fontweight='bold')
ax1.annotate('1825 Peak Depreciation' + NL + '(23.39 Rub/£ = 4.0x Silver)', xy=(1825, 23.39), xytext=(1874, 24.2), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')
ax1.annotate('1839 Kankrin' + NL + 'Reform (6.08)', xy=(1839, 6.08), xytext=(1858, 15.0), arrowprops=dict(arrowstyle='->', color='#1f4e78', lw=0.8), fontsize=8, fontweight='bold', color='#1f4e78')
ax1.set_title("A. St. Petersburg Ruble vs. Metallic Parities (1763-1913)", fontsize=10.5, fontweight='bold')
ax1.set_ylabel("Rubles per Pound Sterling (£)", fontsize=9.5)
ax1.set_ylim(2, 32.0)
ax1.legend(loc='upper left', fontsize=7.8)
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = fig1.add_subplot(2, 2, 2)
u1_sq = u1_15**2
ax2.plot(yrs_1815, u1_sq, color='#1f4e78', lw=1.3)
ax2.fill_between(yrs_1815, 0, u1_sq, color='#1f4e78', alpha=0.2)
for yr, lbl, xy_t in [(1816, '1816 (56.7%)' + NL + 'Post-Waterloo Stabilization', (1842, 0.50)), (1818, '1818 (11.5%)', (1835, 0.16)), (1839, '1839 (3.5%)' + NL + 'Kankrin Reform', (1856, 0.08))]:
    idx_a, = np.where(yrs_1815 == yr)
    v_y = float(u1_sq.item(int(idx_a.item(0))))
    ax2.scatter([yr], [v_y], color='#c00000', s=30, zorder=5)
    ax2.annotate(lbl, xy=(yr, v_y), xytext=xy_t, arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')
ax2.set_title(f"B. European FX Mode 1 Localization: Var1={var1_15:.1f}%, PR(u1)={pr_u1_15:.3f}", fontsize=10.5, fontweight='bold')
ax2.set_ylabel("Squared Temporal Score u1^2 (1816-1913)", fontsize=9.5)
ax2.set_ylim(0, 0.63)
ax2.grid(True, linestyle='--', alpha=0.4)

ax3 = fig1.add_subplot(2, 2, 3)
regimes = ['Assignat Era' + NL + '(1764-1839)', 'Silver & Credit Ruble' + NL + '(1840-1896)', 'Gold Standard' + NL + '(1897-1913)']
rus_shares, neth_shares, germ_shares = [1.87, 14.62, 9.15], [23.05, 23.44, 11.40], [22.50, 1.15, 20.40]
x_r, w_b = np.arange(len(regimes)), 0.25
ax3.bar(x_r - w_b, rus_shares, width=w_b, color='#c00000', label='Russia (St. Petersburg)')
ax3.bar(x_r, neth_shares, width=w_b, color='#1f4e78', label='Netherlands (Amsterdam)')
ax3.bar(x_r + w_b, germ_shares, width=w_b, color='#8eaadb', label='Germany (Hamburg/Berlin)')
ax3.axhline(100.0 / 9.0, color='black', linestyle=':', lw=1.1, label='Uniform Benchmark (1/9 = 11.1%)')
for i, v in enumerate(rus_shares):
    ax3.text(i - w_b, v + 0.6, f"{v:.1f}%", ha='center', fontsize=8, fontweight='bold', color='#800000')
ax3.set_xticks(x_r)
ax3.set_xticklabels(regimes, fontsize=8.5)
ax3.set_ylim(0, 28.5)
ax3.set_ylabel("Share of European Mode 1 Energy (%)", fontsize=9.5)
ax3.set_title("C. Russia's Integration into European FX Core Across Monetary Regimes", fontsize=10.5, fontweight='bold')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, axis='y', linestyle='--', alpha=0.4)

ax4 = fig1.add_subplot(2, 2, 4)
lags = [-2, -1, 0, 1, 2]
rus_g = dln_1815[:, 0]
for idx_c, cname, col, mk in [(1, 'Netherlands -> Russia', '#1f4e78', 'o'), (2, 'Germany -> Russia', '#385723', 's'), (3, 'France -> Russia', '#ed7d31', '^')]:
    other_g = dln_1815[:, idx_c]
    r_vals = [pearsonr(other_g[:lg], rus_g[-lg:])[0] if lg < 0 else (pearsonr(other_g[lg:], rus_g[:-lg])[0] if lg > 0 else pearsonr(other_g, rus_g)[0]) for lg in lags]
    ax4.plot(lags, r_vals, color=col, lw=1.6, marker=mk, label=cname)
ax4.axhline(0, color='black', lw=0.9)
ax4.axhline(1.96 / np.sqrt(len(rus_g)), color='gray', linestyle='--', lw=0.9, label='95% Significance Bound (±0.20)')
ax4.axhline(-1.96 / np.sqrt(len(rus_g)), color='gray', linestyle='--', lw=0.9)
ax4.set_xticks(lags)
ax4.set_xticklabels(['Lag -2y', 'Lag -1y', 'Sync (0)', 'Lag +1y', 'Lag +2y'], fontsize=8.5)
ax4.set_ylim(-0.32, 0.32)
ax4.set_ylabel("Cross-Correlation Coefficient (r)", fontsize=9.5)
ax4.set_title("D. Lead-Lag Arbitrage Test: Annual FX Log-Changes (1816-1913)", fontsize=10.5, fontweight='bold')
ax4.legend(loc='lower left', fontsize=8)
ax4.grid(True, linestyle='--', alpha=0.4)

fig1.suptitle("Invisible Bridges of Credit: St. Petersburg and European Exchange Rate Integration (1763-1913)", fontsize=12.8, fontweight='bold', y=0.98)
plt.savefig('Figure_1_FX.png', bbox_inches='tight')
plt.close(fig1)

# --- FIGURE 2 ---
fig2 = plt.figure(figsize=(15.5, 6.8), dpi=300)
plt.subplots_adjust(wspace=0.26)

ax1 = fig2.add_subplot(1, 2, 1)
for idx_n, cname in enumerate(COUNTRIES_9):
    x_v = float(v1_plot.item(idx_n))
    y_v = float(v2_f.item(idx_n))
    col_pt = '#c00000' if cname == 'Russia' else ('#1f4e78' if cname in ['Netherlands', 'Germany', 'France'] else '#595959')
    ax1.scatter([x_v], [y_v], color=col_pt, s=110 if cname == 'Russia' else 65, zorder=5)
    dx, dy = 0.015, 0.022
    if cname == 'France':
        dx, dy = -0.015, -0.055
    if cname == 'Germany':
        dx, dy = 0.016, -0.015
    if cname == 'Sweden':
        dx, dy = 0.015, -0.035
    ax1.annotate(cname, xy=(x_v, y_v), xytext=(x_v + dx, y_v + dy), fontsize=9, fontweight='bold' if cname in ['Russia', 'Netherlands', 'Germany'] else 'normal', color=col_pt)

ax1.axhline(0, color='black', linestyle=':', lw=0.9)
ax1.axvline(1.0 / np.sqrt(9.0), color='black', linestyle=':', lw=0.9, label='Uniform Mode 1 Loading (1/√9 = 0.333)')
ax1.text(0.18, 0.56, 'Russian Fiscal-Assignat' + NL + 'Outlier Pole', fontsize=8.5, fontweight='bold', color='#800000')
ax1.text(0.43, -0.34, 'Western European' + NL + 'Financial Core', fontsize=8.5, fontweight='bold', color='#1f4e78')
ax1.text(0.11, -0.55, 'Southern European' + NL + 'Periphery', fontsize=8.5, fontweight='bold', color='#595959')
ax1.set_xlim(0.08, 0.64)
ax1.set_ylim(-0.70, 0.68)
ax1.set_xlabel("Mode 1 Core Integration Loading (|v1|, Var1 = 25.7%)", fontsize=9.5)
ax1.set_ylabel("Mode 2 Structural Divergence (v2, Var2 = 11.9%)", fontsize=9.5)
ax1.set_title("A. Structural Map of European Exchange Rate Regimes (1764-1913)", fontsize=10.5, fontweight='bold')
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, linestyle='--', alpha=0.4)

ax2 = fig2.add_subplot(1, 2, 2)
ax2.plot(yrs_p, df_par['ratio_anchor'], color='#c00000', lw=1.8, label='Market Rate / Statutory Metallic Parity')
ax2.axhline(1.0, color='black', linestyle='--', lw=1.1, label='Exact Metallic Parity (Ratio = 1.00)')
ax2.axvspan(1769, 1839, color='#f2dcdb', alpha=0.35, label='Paper Assignat Era (1769-1839)')
ax2.axvspan(1897, 1913, color='#d9e1f2', alpha=0.50, label='Gold Standard Era (1897-1913)')

annotations_b = [
    (1814, '1814 (2.98x)' + NL + 'Napoleonic Wars', 1785, 3.05),
    (1825, '1825 (4.02x)' + NL + 'Assignat Trough', 1792, 4.12),
    (1839, '1839 (1.05x)' + NL + 'Kankrin Stabilization', 1854, 1.76),
    (1878, '1878 (1.68x)' + NL + 'Russo-Turkish War', 1879, 2.48),
    (1905, '1897-1913 (0.99-1.00x)' + NL + 'Witte Gold Anchor', 1892, 0.56)
]
for yr, lbl, tx, ty in annotations_b:
    v_r = float(df_par['ratio_anchor'].loc[yr])
    ax2.scatter([yr], [v_r], color='#800000', s=28, zorder=5)
    ax2.annotate(lbl, xy=(yr, v_r), xytext=(tx, ty), arrowprops=dict(arrowstyle='->', color='#800000', lw=0.8), ha='center', fontsize=8, fontweight='bold', color='#800000')

ax2.set_ylim(0.35, 4.55)
ax2.set_ylabel("Ratio of Market Rate to Metallic Parity", fontsize=9.5)
ax2.set_title("B. Departure and Return to Metallic Anchors in St. Petersburg (1763-1913)", fontsize=10.5, fontweight='bold')
ax2.legend(loc='upper right', fontsize=7.8)
ax2.grid(True, linestyle='--', alpha=0.4)

fig2.suptitle("Fiscal Dominance vs. Monetary Anchors in Imperial Russia's Foreign Exchange", fontsize=12.5, fontweight='bold', y=1.01)
plt.savefig('Figure_2_FX.png', bbox_inches='tight')
plt.close(fig2)
print("Clean pipeline completed! Saved Figure_1_FX.png and Figure_2_FX.png.")


/usr/local/lib/python3.13/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


=== TABLE 1: SPECTRAL SUMMARY ===
Full European Panel (1764-1913, N=9)          | M=150 | Var1=25.7% | PR(u1)=0.095 [0.075, 0.128] | PR(v1)=0.558 [0.464, 0.719] | Russia= 4.32%
Uninterpolated Clean (1816-1913, N=8)         | M= 98 | Var1=30.7% | PR(u1)=0.020 [0.012, 0.084] | PR(v1)=0.428 [0.372, 0.688] | Russia= 3.94%
Assignat and Napoleonic Era (1764-1839)       | M= 76 | Var1=35.5% | PR(u1)=0.158 [0.119, 0.220] | PR(v1)=0.610 [0.557, 0.744] | Russia= 1.87%
Silver and Credit Ruble Era (1840-1896)       | M= 57 | Var1=19.9% | PR(u1)=0.205 [0.114, 0.278] | PR(v1)=0.711 [0.327, 0.764] | Russia=14.62%
Classical Gold Standard (1897-1913)           | M= 17 | Var1=35.2% | PR(u1)=0.346 [0.226, 0.453] | PR(v1)=0.778 [0.523, 0.847] | Russia= 9.15%
Clean pipeline completed! Saved Figure_1_FX.png and Figure_2_FX.png.
